# Script 13 - Linearity of the rhythm-outcome association (Supp Table S10, Supp Fig S3)

Fits each rhythm metric against each outcome twice -- once linear, once with a
natural cubic spline (df = 4) -- and Wald-tests the non-linear basis columns.
A rejection means the spline fits better than a straight line; it does not
identify a threshold.

Panel A tests all three metrics against PHQ-9 (18+) and the cognition
composite (60+). Panel B applies the same test to the first-leg paths of the
structural models -- each metric against KDM-BA advancement and against log
SII -- within the two primary age cells and pooled. Panel C asks whether the
relative-amplitude associations that reject linearity on the raw scale are
linear on the quintile (rank) scale: the quintile score is entered as a
single linear term, and a 3-df Wald test on the quadratic, cubic and quartic
orthogonal contrasts of the score tests for departure from that line; the
quintile-specific contrasts are listed alongside. Estimates are WLS-HC3
without the complex-survey design correction; the notebook checks the
linearity assumption of the Mplus models and does not re-estimate effects.

Reads `data/processed/8_nhanes_analytic.csv`. Writes
`result/supplementary/SuppTableS10_GAM_Linearity.{csv,xlsx}`,
`result/raw/script13_GAM_full.csv`,
`figure/supplementary/SuppFigS3_GAM_Partial_Effects.{pdf,png}` and
`figure/source_data/Source_Data_SuppFigS3_GAM_Partial_Effects.xlsx`.

## Step 0 — Imports and paths

In [ ]:
import os, sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
warnings.filterwarnings('ignore')
pd.options.display.max_columns = 60

def find_root():
    """Locate the project root without assuming where the kernel started."""
    env = os.environ.get('NHANES_SEM_ROOT')
    cands = ([Path(env)] if env else []) + [Path.cwd(), *Path.cwd().parents]
    for q in cands:
        if (q / 'data' / 'processed').is_dir() and (q / 'scripts').is_dir():
            return q
    raise RuntimeError(
        'cannot find the project root from ' + str(Path.cwd())
        + ' -- start Jupyter inside the project, or set NHANES_SEM_ROOT')

BASE   = find_root()
PROC   = BASE / 'data' / 'processed'
R_SUPP = BASE / 'result' / 'supplementary'
R_RAW  = BASE / 'result' / 'raw'
F_SUPP = BASE / 'figure' / 'supplementary'
F_SRC  = BASE / 'figure' / 'source_data'
for q in (R_SUPP, R_RAW, F_SUPP, F_SRC):
    q.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(BASE / 'scripts'))

# Reload project modules so the notebook runs against the code on disk.
import importlib
import wls_utils, nature_palette
import population; importlib.reload(population)
for _m in (wls_utils, nature_palette,):
    importlib.reload(_m)

# chi2_sf comes from wls_utils; scipy is not a dependency.
from wls_utils import wls_fit, ns_basis, ns_knots, chi2_sf
from population import ADULT_MIN_AGE, OLD_MIN_AGE, YOUNG, OLD, POOLED
from nature_palette import PHQ9_COLOR, COG_COLOR, GREY
from nature_palette import fmt_ci as _fmt_ci, age_label

from nature_palette import EXP_COL, EXP_ORDER  # one source of truth
EXPOSURES = list(EXP_ORDER)
EXP_NAME  = {'IV': 'Intradaily variability', 'IS': 'Interdaily stability',
             'RA': 'Relative amplitude'}


## Step 1 — Load + rename + derive PHQ9 / COG_Z

In [ ]:
df = pd.read_csv(PROC / '8_nhanes_analytic.csv', low_memory=False)
rename = {
    'RIDAGEYR':       'AGE',
    'RIAGENDR':       'SEX',
    'BMXBMI':         'BMI',
    'DMDEDUC2':       'EDU',
    'Wear_Time':      'WEAR_T',
    'SLD010H':        'SLEEP_H',
}
df = df.rename(columns=rename)
assert 'WTMEC4YR' in df.columns, 'run script 8: WTMEC4YR missing'
assert np.allclose(df['WTMEC4YR'], df['WTMEC2YR'] / 2.0, equal_nan=True), \
    'stored WTMEC4YR does not match WTMEC2YR / 2 -- the file is stale'
df = df[df['AGE'] >= ADULT_MIN_AGE].copy()   # the analysis population (population.py)

dpq_cols = [f'DPQ{i:03d}' for i in [10, 20, 30, 40, 50, 60, 70, 80, 90]]
df['PHQ9'] = df[dpq_cols].replace({7: np.nan, 9: np.nan}).sum(axis=1, min_count=9)

elig_60 = df['AGE'] >= OLD_MIN_AGE
for src, dst in [('memory_score', 'MEM_Z'),
                 ('fluency_score', 'FLUEN_Z'),
                 ('speed_score',  'SPEED_Z')]:
    raw = df[src]
    s = raw.loc[elig_60].dropna()
    df[dst] = np.where(elig_60, (raw - s.mean()) / s.std(), np.nan)
df['COG_Z'] = df[['MEM_Z', 'FLUEN_Z', 'SPEED_Z']].mean(axis=1, skipna=False)

print(f'Adults {ADULT_MIN_AGE}+: n = {len(df):,}')
print(f'  PHQ-9 non-null: {df["PHQ9"].notna().sum():,}')
print(f'  COG_Z non-null (60+): {df["COG_Z"].notna().sum():,}')

## Step 2 — Helpers: per-outcome sample + survey-weighted standardisation

In [ ]:
M1_COVS = ['AGE', 'SEX', 'BMI', 'EDU', 'WEAR_T', 'SLEEP_H']

def fmt_p(p):
    return '<0.001' if p < 0.001 else f'{p:.3f}'

def fmt_ci(est, lo, hi, dp=3):
    return _fmt_ci(est, lo, hi, dp)   # minus kept, plus omitted

def weighted_mean_sd(x, w):
    w_tot = np.sum(w)
    mu = np.sum(x * w) / w_tot
    var = np.sum(w * (x - mu) ** 2) / w_tot
    return mu, float(np.sqrt(var))

def per_outcome_sample(exposure, outcome_col, age_min=ADULT_MIN_AGE, age_max=None):
    needed = [exposure, outcome_col, 'WTMEC4YR'] + M1_COVS
    mask = (df['AGE'] >= age_min) & df[needed].notna().all(axis=1)
    if age_max is not None:
        mask &= df['AGE'] < age_max
    return df.loc[mask].copy()

## Step 3 — Fit linear + GAM, Wald chi-square nonlinearity test

For each metric and outcome:
1. Standardise the metric and the outcome using survey-weighted SDs on the sample.
2. Linear model: covariates + metric_z.
3. GAM: covariates + ns_basis(metric_z, df = 4) (4 columns = 1 linear + 3 smooth).
4. Wald χ² test on the three smooth basis coefficients.
5. GAM partial-dependence curve on a grid from the 1st to the 99th percentile of the metric.

In [ ]:
def fit_linear_and_gam(exposure, outcome_col, age_min=ADULT_MIN_AGE, age_max=None, df_spline=4):
    """Linear vs natural-cubic-spline fit for one rhythm metric and outcome; a
    rejection means the spline fits better than a straight line, not that a
    threshold exists."""
    ana = per_outcome_sample(exposure, outcome_col, age_min=age_min, age_max=age_max)
    w = ana['WTMEC4YR'].values
    mu_x, sd_x = weighted_mean_sd(ana[exposure].values,    w)
    mu_y, sd_y = weighted_mean_sd(ana[outcome_col].values, w)
    x_z = (ana[exposure].values    - mu_x) / sd_x
    y_z = (ana[outcome_col].values - mu_y) / sd_y
    cov_mat = np.column_stack([ana[c].values for c in M1_COVS])

    X_lin = np.column_stack([x_z, cov_mat])
    fit_lin = wls_fit(X_lin, y_z, w=w, se='hc3', add_const=True)

    # Knots fixed once on the fitting sample and reused for the prediction
    # grid and the reference point (identical to ns_basis's default on x_z,
    # so the fit and the Wald test are unchanged).
    knots, bound = ns_knots(x_z, df_spline)
    B_x = ns_basis(x_z, df=df_spline, knots=knots, boundary=bound)
    X_gam = np.column_stack([B_x, cov_mat])
    fit_gam = wls_fit(X_gam, y_z, w=w, se='hc3', add_const=True)

    # Wald test on the non-linear basis columns (cols 2..df_spline; col 1 is
    # the linear direction inside the basis).
    smooth_idx = list(range(2, 1 + df_spline))
    beta_s = fit_gam['params'][smooth_idx]
    cov_s = fit_gam['cov'][np.ix_(smooth_idx, smooth_idx)]
    W = float(beta_s.T @ np.linalg.pinv(cov_s) @ beta_s)
    df_smooth = len(smooth_idx)
    p_nonlin = chi2_sf(W, df_smooth)

    grid_raw = np.linspace(np.percentile(ana[exposure], 1),
                           np.percentile(ana[exposure], 99), 200)
    grid_z = (grid_raw - mu_x) / sd_x
    B_grid = ns_basis(grid_z, df=df_spline, knots=knots, boundary=bound)
    # Difference from the weighted-mean exposure (z = 0), where the linear
    # reference is zero by construction: d(x) = B(x) - B(x_ref); curve d(x)b,
    # standard error sqrt(d(x) V d(x)') with the HC3 covariance V of the
    # spline coefficients. Curve and interval are both zero at x_ref.
    B_ref = ns_basis(np.array([0.0]), df=df_spline, knots=knots, boundary=bound)
    D = B_grid - B_ref
    x_idx = list(range(1, 1 + df_spline))
    V = fit_gam['cov'][np.ix_(x_idx, x_idx)]
    f_pdep = D @ fit_gam['params'][x_idx]
    se_d = np.sqrt(np.maximum(np.einsum('ij,jk,ik->i', D, V, D), 0))
    lo_pdep, hi_pdep = f_pdep - 1.96 * se_d, f_pdep + 1.96 * se_d
    return {
        'exposure': exposure, 'outcome': outcome_col, 'n': int(fit_lin['n']),
        'beta_linear': float(fit_lin['params'][1]),
        'ci_lo_linear': float(fit_lin['ci_lo'][1]),
        'ci_hi_linear': float(fit_lin['ci_hi'][1]),
        'p_linear': float(fit_lin['p'][1]),
        'r2_linear': float(fit_lin['r2']), 'r2_gam': float(fit_gam['r2']),
        'delta_r2': float(fit_gam['r2'] - fit_lin['r2']),
        'wald_chi2': W, 'df_smooth': df_smooth, 'p_nonlin': p_nonlin,
        'grid_raw': grid_raw, 'grid_z': grid_z,
        'x_ref_raw': mu_x, 'sd_x_raw': sd_x,
        'knots_z': knots, 'boundary_z': np.asarray(bound),
        'f_pdep': f_pdep, 'lo_pdep': lo_pdep, 'hi_pdep': hi_pdep,
        'f_lin': float(fit_lin['params'][1]) * grid_z,
    }

## Step 4 — Run for Depression and Cognition

In [ ]:
# Every rhythm metric, both outcomes.
RES = {}
for exp in EXPOSURES:
    RES[(exp, 'PHQ9')]  = fit_linear_and_gam(exp, 'PHQ9',  age_min=ADULT_MIN_AGE)
    RES[(exp, 'COG_Z')] = fit_linear_and_gam(exp, 'COG_Z', age_min=60)

for (exp, out), R in RES.items():
    lab = f'{EXP_NAME[exp]} -> ' + ('PHQ-9 (\u226518)' if out == 'PHQ9' else 'cognition (60+)')
    verdict = 'NON-LINEAR' if R['p_nonlin'] < 0.05 else 'linear supported'
    print(f'{lab:52s} n={R["n"]:>6,}  linear beta={R["beta_linear"]:.3f} '
          f'(p={fmt_p(R["p_linear"])})   Wald chi2({R["df_smooth"]})={R["wald_chi2"]:6.2f} '
          f'p_nonlin={fmt_p(R["p_nonlin"])}  -> {verdict}')


## Step 5 — Build Supp Table S10

In [ ]:
rows = []
for (exp, out), R in RES.items():
    rows.append({
        'Rhythm metric': f'{EXP_NAME[exp]} ({exp})',
        'Outcome': f'PHQ-9 (\u2265{ADULT_MIN_AGE})' if out == 'PHQ9' else 'Cognition composite (\u226560)',
        'n': f"{R['n']:,}",
        'Linear beta (95% CI)': fmt_ci(R['beta_linear'], R['ci_lo_linear'], R['ci_hi_linear']),
        'Linear p': fmt_p(R['p_linear']),
        'R2 linear': f"{R['r2_linear']:.4f}",
        'R2 GAM': f"{R['r2_gam']:.4f}",
        'delta R2': f"{R['delta_r2']:.5f}",
        'Wald chi2 (df=3)': f"{R['wald_chi2']:.3f}",
        'p (non-linearity)': fmt_p(R['p_nonlin']),
        'Verdict': 'Departs from linearity' if R['p_nonlin'] < 0.05 else 'No clear evidence of non-linearity',
    })
tableS10 = pd.DataFrame(rows)
print(tableS10.to_string(index=False))
tableS10.to_csv(R_SUPP / 'SuppTableS10_GAM_Linearity.csv', index=False)
pd.DataFrame([{k: v for k, v in R.items()
               if not isinstance(v, np.ndarray)} for R in RES.values()]
             ).to_csv(R_RAW / 'script13_GAM_full.csv', index=False)

NOTES = [
    'Supportive diagnostics: survey-weighted least squares with HC3 '
    'standard errors, without the NHANES strata and primary sampling units. '
    'They check the functional form assumed by the complex-survey structural '
    'models and are not used for inference about effects.',
    'Spline: a natural cubic spline in the z-standardised metric with four '
    'basis functions, parameterised as in Hastie, Tibshirani and Friedman '
    '(Elements of Statistical Learning, eq. 5.4-5.5): the linear term x and '
    'three truncated-power functions, with boundary knots at the 5th and 95th '
    'percentiles of the metric and three interior knots at its quartiles (25th, '
    '50th and 75th percentiles; wls_utils.ns_basis). The restricted model is '
    'the straight line in x; the full model is the same sample and covariates '
    'with the three spline functions added; the Wald test (3 df, HC3 '
    'covariance) is on their coefficients.',
    'delta R2 is the gain in R2 of the spline model over the linear model on '
    'the same sample; it is not the unique-variance delta R2 of '
    'Supplementary Fig. S6.',
    'A test that rejects means the spline fits better than a straight line '
    'in raw units; it does not identify a threshold. A test that does not '
    'reject is reported as no clear evidence of non-linearity, not as proof '
    'of linearity.',
    'Covariates: age, sex, BMI, education, wear time, sleep duration.',
]
with pd.ExcelWriter(R_SUPP / 'SuppTableS10_GAM_Linearity.xlsx',
                    engine='openpyxl') as xw:
    tableS10.to_excel(xw, sheet_name='Supp Table S10', index=False)
    pd.DataFrame({'Notes': NOTES}).to_excel(xw, sheet_name='Notes', index=False)
print(f"\nwrote {R_SUPP / 'SuppTableS10_GAM_Linearity.xlsx'}")

for stale in [R_SUPP / 'SuppTableS5_GAM_Linearity.csv',
              R_SUPP / 'SuppTableS5_GAM_Linearity.xlsx']:
    if stale.exists():
        dest = BASE / '_superseded' / stale.name
        dest.parent.mkdir(exist_ok=True)
        stale.rename(dest)
        print(f'superseded -> _superseded/{stale.name}')


## Step 5b — Panel B: the first-leg paths (rhythm → biological measure), by age cell

The structural models' central estimates are the a1 (KDM-BA advancement) and
a2 (log SII) paths within the 18–59 and ≥60 cells. The same linear-versus-
spline test is applied to those paths; the pooled adult sample is added for
reference. The dependent variables are left on their natural scale and
standardised within each sample, as in Panel A.

In [ ]:
BIO = [('KDM_BA_advance', 'KDM-BA advancement'), ('SII_log', 'log SII')]
CELLS_B = [(YOUNG, ADULT_MIN_AGE, OLD_MIN_AGE), (OLD, OLD_MIN_AGE, None), (POOLED, ADULT_MIN_AGE, None)]

RES_B = {}
for exp in EXPOSURES:
    for col, lab in BIO:
        for cell_name, lo, hi in CELLS_B:
            RES_B[(exp, col, cell_name)] = fit_linear_and_gam(exp, col, age_min=lo, age_max=hi)

rows_b = []
for (exp, col, cell_name), R in RES_B.items():
    rows_b.append({
        'Rhythm metric': f'{EXP_NAME[exp]} ({exp})',
        'Dependent variable': dict(BIO)[col],
        'Age group': age_label(cell_name),
        'n': f"{R['n']:,}",
        'Linear beta (95% CI)': fmt_ci(R['beta_linear'], R['ci_lo_linear'], R['ci_hi_linear']),
        'Linear p': fmt_p(R['p_linear']),
        'R2 linear': f"{R['r2_linear']:.4f}",
        'R2 GAM': f"{R['r2_gam']:.4f}",
        'delta R2': f"{R['delta_r2']:.5f}",
        'Wald chi2 (df=3)': f"{R['wald_chi2']:.3f}",
        'p (non-linearity)': fmt_p(R['p_nonlin']),
        'Verdict': 'Departs from linearity' if R['p_nonlin'] < 0.05 else 'No clear evidence of non-linearity',
    })
tableS10B = pd.DataFrame(rows_b)
print(tableS10B.to_string(index=False))

## Step 5c — Panel C: are the relative-amplitude associations linear on the quintile scale?

Relative amplitude is compressed and left-skewed (median 0.88, 0.2% at the
ceiling of 1), so a straight line in raw units need not be a straight line in
population rank. The quintile score (1–5) is entered as a single linear term,
and the quadratic, cubic and quartic orthogonal (Legendre) contrasts of the
score are added; a 3-df Wald test on those contrasts tests departure from
linearity in the score. The four quintile contrasts (Q2–Q5 against Q1, in
standard deviations of the dependent variable) show the shape directly.

In [ ]:
def quintile_scale_test(exposure, outcome_col, age_min=ADULT_MIN_AGE, age_max=None):
    ana = per_outcome_sample(exposure, outcome_col, age_min=age_min, age_max=age_max)
    w = ana['WTMEC4YR'].values
    q = (pd.qcut(ana[exposure], 5, labels=False).values + 1).astype(float)
    cov_mat = np.column_stack([ana[c].values for c in M1_COVS])
    mu_y, sd_y = weighted_mean_sd(ana[outcome_col].values, w)
    y_z = (ana[outcome_col].values - mu_y) / sd_y
    # orthogonal (Legendre) contrasts of the score on [-1, 1]: degrees 1-4
    P = np.polynomial.legendre.legvander((q - 3.0) / 2.0, 4)[:, 1:]
    fit_score = wls_fit(np.column_stack([P[:, :1], cov_mat]), y_z, w=w, se='hc3', add_const=True)
    fit_poly = wls_fit(np.column_stack([P, cov_mat]), y_z, w=w, se='hc3', add_const=True)
    idx = [2, 3, 4]                       # quadratic, cubic, quartic (col 0 is the intercept)
    b = fit_poly['params'][idx]
    V = fit_poly['cov'][np.ix_(idx, idx)]
    W = float(b.T @ np.linalg.pinv(V) @ b)
    D = np.column_stack([(q == k).astype(float) for k in range(2, 6)])
    fit_cat = wls_fit(np.column_stack([D, cov_mat]), y_z, w=w, se='hc3', add_const=True)
    return {
        'n': int(fit_score['n']),
        'beta_score': float(fit_score['params'][1] / 2.0),   # per quintile step (Legendre degree 1 = (q-3)/2)
        'r2_score': float(fit_score['r2']), 'r2_dummies': float(fit_cat['r2']),
        'wald_chi2': W, 'p_nonlin_quintile': chi2_sf(W, 3),
        'contrasts': fit_cat['params'][1:5],
    }

SPECS_C = [('PHQ9', 'PHQ-9', ADULT_MIN_AGE, None), ('COG_Z', 'Cognition composite', OLD_MIN_AGE, None),
           ('KDM_BA_advance', 'KDM-BA advancement', ADULT_MIN_AGE, OLD_MIN_AGE),
           ('KDM_BA_advance', 'KDM-BA advancement', OLD_MIN_AGE, None),
           ('SII_log', 'log SII', ADULT_MIN_AGE, OLD_MIN_AGE), ('SII_log', 'log SII', OLD_MIN_AGE, None)]
rows_c = []
for col, lab, lo, hi in SPECS_C:
    R = quintile_scale_test('RA', col, age_min=lo, age_max=hi)
    rows_c.append({
        'Rhythm metric': 'Relative amplitude (RA)',
        'Dependent variable': lab,
        'Age group': age_label(YOUNG if (lo, hi) == (ADULT_MIN_AGE, OLD_MIN_AGE) else OLD if lo == OLD_MIN_AGE else POOLED),
        'n': f"{R['n']:,}",
        'beta per quintile step (SD of dependent variable)': f"{R['beta_score']:.3f}",
        'R2, quintile score as a linear term': f"{R['r2_score']:.4f}",
        'R2, four quintile indicators': f"{R['r2_dummies']:.4f}",
        'Wald chi2 (df=3), departure from linearity in the score': f"{R['wald_chi2']:.3f}",
        'p': fmt_p(R['p_nonlin_quintile']),
        'Q2 vs Q1': f"{R['contrasts'][0]:.3f}", 'Q3 vs Q1': f"{R['contrasts'][1]:.3f}",
        'Q4 vs Q1': f"{R['contrasts'][2]:.3f}", 'Q5 vs Q1': f"{R['contrasts'][3]:.3f}",
    })
tableS10C = pd.DataFrame(rows_c)
print(tableS10C.to_string(index=False))

NOTES_BC = [
    'PANEL B. The same linear-versus-spline test applied to the first-leg paths '
    'of the structural models: each rhythm metric against KDM-BA advancement '
    f'and against log SII, within the primary age cells ({age_label(YOUNG)}, {OLD_MIN_AGE} and over) and '
    'pooled. Survey-weighted least squares with HC3 errors, the dependent '
    'variable standardised within each sample; covariates as in Panel A.',
    'PANEL C. Linearity of the relative-amplitude associations on the quintile '
    'scale. The RA quintile score (1-5; unweighted quintiles of the observed '
    'values within each analysis sample, so cut points differ between samples) is '
    'entered as a single linear term; the quadratic, cubic and quartic '
    'orthogonal contrasts of the score are added and tested jointly (3 df). A '
    'non-significant test means that the four quintile indicators do not fit '
    'better than the straight line through the quintile score. Q2-Q5 against Q1 '
    'are the quintile indicators\' coefficients, in standard deviations of the '
    'dependent variable, from the model with four indicators.',
    'Relative amplitude is compressed and left-skewed (median 0.88; 0.2% of '
    'values at the ceiling of 1), so equal steps in raw units cover very '
    'different shares of the population; a straight line in raw units and a '
    'straight line in population rank are different hypotheses, and Panels A-B '
    'and Panel C test them separately.',
]
with pd.ExcelWriter(R_SUPP / 'SuppTableS10_GAM_Linearity.xlsx', engine='openpyxl') as xw:
    tableS10.to_excel(xw, sheet_name='Supp Table S10', index=False)
    tableS10B.to_excel(xw, sheet_name='S10 Panel B biological', index=False)
    tableS10C.to_excel(xw, sheet_name='S10 Panel C quintile scale', index=False)
    pd.DataFrame({'Notes': ['PANEL A. ' + NOTES[0]] + NOTES[1:] + NOTES_BC}).to_excel(
        xw, sheet_name='Notes', index=False)
print(f"\nrewrote {R_SUPP / 'SuppTableS10_GAM_Linearity.xlsx'} with Panels A-C")

## Step 6 — Supp Figure S3: GAM partial-effect plots

In [ ]:
mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Helvetica', 'Arial', 'DejaVu Sans'],
    'font.size': 8, 'axes.titlesize': 8, 'axes.labelsize': 7.5,
    'xtick.labelsize': 7, 'ytick.labelsize': 7, 'legend.fontsize': 7,
    'axes.linewidth': 0.5, 'lines.linewidth': 0.9,
    'pdf.fonttype': 42, 'ps.fonttype': 42, 'savefig.dpi': 600,
    'axes.spines.top': False, 'axes.spines.right': False,
})
MM = 1 / 25.4

fig, axes = plt.subplots(2, 3, figsize=(183 * MM, 112 * MM), sharex='col')
for j, exp in enumerate(EXPOSURES):
    for i, (out, olab, ocol) in enumerate([
            ('PHQ9', f'PHQ-9 (SD)\nadults \u2265{ADULT_MIN_AGE} years', PHQ9_COLOR),
            ('COG_Z', f'cognitive composite (SD)\nadults \u2265{OLD_MIN_AGE} years', COG_COLOR)]):
        ax = axes[i, j]
        R = RES[(exp, out)]
        ax.fill_between(R['grid_raw'], R['lo_pdep'], R['hi_pdep'],
                        color=EXP_COL[exp], alpha=0.15, lw=0)
        ax.plot(R['grid_raw'], R['f_pdep'], color=EXP_COL[exp], lw=1.1)
        ax.plot(R['grid_raw'], R['f_lin'], color=GREY, lw=0.8, ls='--')
        ax.axhline(0, color='0.4', lw=0.4, ls=':')
        _p = fmt_p(R['p_nonlin'])
        ax.text(0.03, 0.94,
                'P for nonlinearity ' + (_p if _p.startswith('<') else '= ' + _p),
                transform=ax.transAxes, va='top', fontsize=7, zorder=6,
                bbox=dict(facecolor='white', edgecolor='none', alpha=0.85, pad=1))
        ax.text(-0.02 if j == 0 else -0.02, 1.02, 'abcdef'[3 * i + j],
                transform=ax.transAxes, ha='right', va='bottom',
                fontsize=10, fontweight='bold')
        if i == 0:
            ax.set_title(f'{EXP_NAME[exp]} ({exp})', color=EXP_COL[exp], pad=3)
        if i == 1:
            ax.set_xlabel(f'{exp} (raw scale)')
        if j == 0:
            ax.set_ylabel(f'Difference in {olab}', color=ocol)
fig.tight_layout()

stem = 'SuppFigS3_GAM_Partial_Effects'
fig.savefig(F_SUPP / f'{stem}.pdf', bbox_inches='tight')
fig.savefig(F_SUPP / f'{stem}.png', bbox_inches='tight', dpi=600)
plt.close(fig)
print(f'wrote {F_SUPP / stem}.pdf + .png')

cap = ['Supplementary Fig. S3 | Linearity of each rhythm-outcome association.',
       'a-c, depressive symptoms (PHQ-9), adults aged \u2265' + str(ADULT_MIN_AGE) + ' years; '
       'd-f, observed cognitive composite, adults aged \u2265' + str(OLD_MIN_AGE) + ' years; '
       'columns relative amplitude, interdaily stability and intradaily variability, '
       'on their raw scales.',
       'Solid line: difference in the standardised outcome from its value at the '
       'survey-weighted mean of the metric, from the natural-spline model '
       '(4 basis functions; knots fixed on the fitting sample, as in '
       'Supplementary Table 10), with a pointwise 95% confidence band computed '
       'from the HC3 covariance of the basis differences B(x) - B(mean); curve '
       'and band are zero at the mean. Dashed grey: the linear fit, also zero at '
       'the mean.',
       'P for nonlinearity is the 3-df Wald test of the non-linear spline terms '
       '(Supplementary Table 10). A small P means the spline fits better than a '
       'straight line; it does not identify a threshold.',
       'Survey-weighted least squares with HC3 errors, adjusted for age, sex, '
       'body mass index, education, accelerometer wear time and sleep duration; '
       'no complex-survey design correction. These are supportive diagnostics; '
       'inference follows the structural models.']
with pd.ExcelWriter(F_SRC / f'Source_Data_{stem}.xlsx', engine='openpyxl') as xw:
    for (exp, out), R in RES.items():
        sheet = f"{exp} {'PHQ9' if out == 'PHQ9' else 'COG'}"
        pd.DataFrame({
            f'{exp}_raw': R['grid_raw'],
            'spline_difference_from_mean': R['f_pdep'],
            'spline_CI_lo': R['lo_pdep'], 'spline_CI_hi': R['hi_pdep'],
            'linear_reference': R['f_lin'],
            'reference_raw_mean': R['x_ref_raw'],
            'P_nonlinearity': R['p_nonlin'],
        }).to_excel(xw, sheet_name=sheet, index=False)
    pd.DataFrame({'Notes': cap}).to_excel(xw, sheet_name='legend', index=False)
print(f'wrote {F_SRC / ("Source_Data_" + stem + ".xlsx")}')

for stale in [F_SUPP / f'{stem}.tiff', F_SRC / 'Source_Data_SuppFigS3.xlsx']:
    if stale.exists():
        dest = BASE / '_superseded' / stale.name
        dest.parent.mkdir(exist_ok=True)
        stale.rename(dest)
        print(f'superseded -> _superseded/{stale.name}')
